# Train Network Map Viewer

In this notebook, you will visualize the train network simulation on an interactive map:

1. Display all stations with color-coding (entry stations = blue, exit stations = green)
2. Show passenger queue sizes as circles around each station
3. Display train positions as they move through the network
4. Show train capacity as percentage filled

This provides real-time visibility into the simulation state.

## Step 1: Setup and Configuration

Load configuration and create the base map centered on the train network.

In [ ]:
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector
from anymap_ts import Map
import json

# Load configuration
config = load_config()

# Calculate center point of the network
stations = config.train_network.route
avg_lat = sum(s.location.lat for s in stations) / len(stations)
avg_lon = sum(s.location.lon for s in stations) / len(stations)

print(f"Train network has {len(stations)} stations")
print(f"Center: {avg_lat:.4f}, {avg_lon:.4f}")

## Step 2: Create Interactive Map

Initialize the map with the train network visible.

In [ ]:
# Create map centered on the network
map_widget = Map(
    center=(avg_lon, avg_lat),
    zoom=14,
    style="https://basemaps.cartocdn.com/gl/positron-gl-style/style.json"
)

map_widget

## Step 3: Add Station Markers

Display all stations with different colors for entry vs exit stations.

In [ ]:
# Color scheme
ENTRY_COLOR = "#2196F3"  # Blue
EXIT_COLOR = "#4CAF50"   # Green

# Add station markers
for station in stations:
    color = ENTRY_COLOR if station.type == "entry" else EXIT_COLOR
    
    map_widget.add_marker(
        lnglat=(station.location.lon, station.location.lat),
        color=color,
        text=station.name,
        draggable=False
    )

print(f"✓ Added {len(stations)} station markers")
print(f"  Blue = Entry stations")
print(f"  Green = Exit stations")

## Step 4: Simulate Passenger Queues

Create visual indicators for passenger waiting at each station using circles.

In [ ]:
from simulated_city.agents import Station, SimulationState, Passenger
from datetime import datetime

# Initialize simulation state
sim_state = SimulationState(current_time=datetime.now())

# Convert config to Station objects
route = []
for station_config in config.train_network.route:
    station = Station(
        name=station_config.name,
        station_type=station_config.type,
        location_lat=station_config.location.lat,
        location_lon=station_config.location.lon,
        exit_percentage=station_config.exit_percentage,
    )
    route.append(station)

entry_stations = [s for s in route if s.station_type == "entry"]

print(f"Monitoring {len(entry_stations)} entry stations")

In [ ]:
# Add passengers to queues for visualization
queue_sizes = [180, 280, 150]  # Different loads at each entry station

for i, station in enumerate(entry_stations):
    queue = sim_state.get_station_queue(station.name)
    passenger_count = queue_sizes[i]
    
    # Add passengers
    passengers = [
        Passenger(f"{station.name}-p-{j}", station.name, route[-1].name, datetime.now())
        for j in range(passenger_count)
    ]
    queue.add_passengers(passengers)
    
    print(f"{station.name}: {passenger_count} passengers waiting")

## Step 5: Visualize Queue Sizes as Circles

Draw circles around each entry station sized proportionally to the queue.

In [ ]:
# GeoJSON for queue visualization circles
features = []

for station in entry_stations:
    queue = sim_state.get_station_queue(station.name)
    waiting_count = queue.count
    
    # Calculate circle radius based on queue size (meters)
    # Scale: 1 passenger = ~0.5 meters radius
    radius_meters = max(20, waiting_count * 0.5)
    
    # Create circle as GeoJSON point with style
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [station.location_lon, station.location_lat]
        },
        "properties": {
            "title": f"{station.name}",
            "waiting": waiting_count,
            "radius": radius_meters,
            "color": "#FF9800"  # Orange for queues
        }
    }
    features.append(feature)

geojson_queues = {
    "type": "FeatureCollection",
    "features": features
}

print(f"Created {len(features)} queue visualizations")
print("\nQueue sizes:")
for feature in features:
    props = feature["properties"]
    print(f"  {props['title']}: {props['waiting']} passengers (radius: {props['radius']:.1f}m)")

## Step 6: Add Train Positions

Show trains at different positions along the route with capacity indicators.

In [ ]:
from simulated_city.agents import Train, TrainAgent

# Create sample trains at different positions and capacities
train_positions = [
    {"train_id": "T1", "station_idx": 0, "current": 80, "capacity": 180},   # 44% full
    {"train_id": "T2", "station_idx": 2, "current": 150, "capacity": 180},  # 83% full
    {"train_id": "T3", "station_idx": 3, "current": 45, "capacity": 180},   # 25% full
]

train_markers = []

for train_info in train_positions:
    station = route[train_info["station_idx"]]
    capacity_percent = int((train_info["current"] / train_info["capacity"]) * 100)
    
    # Color based on capacity
    if capacity_percent < 50:
        color = "#4CAF50"  # Green - plenty of space
    elif capacity_percent < 80:
        color = "#FF9800"  # Orange - getting full
    else:
        color = "#F44336"  # Red - nearly full
    
    # Offset train position slightly so it doesn't overlap station marker
    offset_lon = station.location_lon + 0.001
    offset_lat = station.location_lat + 0.001
    
    map_widget.add_marker(
        lnglat=(offset_lon, offset_lat),
        color=color,
        text=f"{train_info['train_id']}: {capacity_percent}% full",
        draggable=False
    )
    
    train_markers.append(train_info)
    print(f"✓ {train_info['train_id']} at {station.name}: {capacity_percent}% capacity")

## Step 7: Add Route Line

Connect stations with a line showing the train route.

In [ ]:
# Create route line GeoJSON
route_coords = [[s.location.lon, s.location.lat] for s in route]

route_geojson = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": route_coords
        },
        "properties": {
            "name": "Train Route",
            "color": "#9E9E9E"
        }
    }]
}

# Add to map (this would require adding GeoJSON layer support to anymap-ts)
# For now, we'll export it for reference
print("✓ Route line prepared")
print(f"  Connects {len(route)} stations")

## Step 8: Real-time MQTT Subscription (Optional)

Subscribe to train and sensor updates to refresh the map in real-time.

In [ ]:
# Example: Subscribe to sensor updates
connector = MqttConnector(config.mqtt, client_id_suffix="map-viewer")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
    
    # Subscribe to all sensor updates
    sensor_topic = f"{config.train_network.mqtt_base_topic}/station/+/sensor/waiting_count"
    
    # Store latest queue sizes
    latest_queues = {}
    
    def on_sensor_update(client, userdata, message):
        """Handle sensor data updates."""
        try:
            data = json.loads(message.payload.decode())
            station_name = data["station_name"]
            waiting_count = data["waiting_count"]
            
            latest_queues[station_name] = waiting_count
            print(f"  📊 {station_name}: {waiting_count} waiting")
            
            # In a full implementation, update map marker sizes here
            
        except Exception as e:
            print(f"Error processing sensor message: {e}")
    
    connector.client.message_callback_add(sensor_topic, on_sensor_update)
    connector.client.subscribe(sensor_topic, qos=0)
    
    print(f"✓ Subscribed to: {sensor_topic}")
    print("  Waiting for updates... (run simulation in another notebook)")
else:
    print("✗ Failed to connect to MQTT broker")

## Step 9: Export Map State

Save the current visualization state for analysis.

In [ ]:
# Export visualization data
viz_data = {
    "timestamp": datetime.now().isoformat(),
    "stations": [
        {
            "name": s.name,
            "type": s.station_type,
            "location": {"lat": s.location_lat, "lon": s.location_lon},
            "queue_size": sim_state.get_station_queue(s.name).count if s.station_type == "entry" else 0
        }
        for s in route
    ],
    "trains": train_markers,
    "network_stats": {
        "total_waiting": sum(sim_state.get_station_queue(s.name).count for s in entry_stations),
        "trains_deployed": len(train_markers),
        "stations": len(route)
    }
}

print("Visualization State:")
print(f"  Total passengers waiting: {viz_data['network_stats']['total_waiting']}")
print(f"  Trains deployed: {viz_data['network_stats']['trains_deployed']}")
print(f"  Stations: {viz_data['network_stats']['stations']}")

# Could save to file
# import json
# with open('train_map_state.json', 'w') as f:
#     json.dump(viz_data, f, indent=2)

## Cleanup

In [ ]:
# Disconnect from MQTT
if connector.client and connector.client.is_connected():
    connector.disconnect()
    print("✓ Disconnected from MQTT broker")

## Exercise: Interactive Visualization

Try these experiments:

1. **Modify queue sizes**: Change the `queue_sizes` array in Step 4 to see different load patterns
2. **Add more trains**: Add additional entries to `train_positions` with different capacities
3. **Real-time monitoring**: Run `05_train_full_simulation.ipynb` in another window and watch sensor updates
4. **Color thresholds**: Adjust the capacity percentage thresholds for train colors (currently 50% and 80%)

Example for experiment 1:
```python
# Set all stations to critical load
queue_sizes = [350, 380, 320]  # All above threshold

# Re-run Steps 4-5 to update visualization
```

Example for experiment 2:
```python
# Add an extra train
train_positions.append({
    "train_id": "T4",
    "station_idx": 1,
    "current": 100,
    "capacity": 180
})

# Re-run Step 6 to add to map
```

## Next Steps

Continue to **05_train_dashboard.ipynb** to see time-series metrics and performance analysis.